# Silver Layer Processing - Provider Data
## Healthcare Lakehouse Project

This notebook transforms the raw Bronze provider data into a clean, standardized **Silver layer** with proper data modeling.

### Transformations Applied:
1. **Data Cleaning**: Remove metadata columns, handle null/invalid values
2. **Type Casting**: Convert columns to appropriate types (INT, DOUBLE)
3. **Standardization**: Lowercase columns, uppercase text values
4. **Deduplication**: Remove duplicate records
5. **Data Modeling**: Split into dimension (provider) and fact (prescriptions) tables

### Output Tables:
- **silver.provider**: Provider dimension table
- **silver.fact_prescriptions**: Prescription facts table (partitioned by state)

### Step 1: Import Required Libraries

In [ ]:
# Databricks notebook source
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Step 2: Load Utility Variables
Runs the utilities notebook to access schema name variables.

In [ ]:
# MAGIC %run /Workspace/Users/pawanvirat32@gmail.com/healthcare_lakehouse_project/01_setup/utilities

### Step 3: Define Widget Parameters
Creates interactive widgets for runtime configuration:
- **catalog**: Unity Catalog name (default: healthcare_lakehouse)
- **data_source**: Source table name (default: by_provider)

In [ ]:
dbutils.widgets.text('catalog', 'healthcare_lakehouse', 'Catalog')
dbutils.widgets.text('data_source', 'by_provider', 'Data Source')

### Step 4: Get Widget Values

In [ ]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(f"catalog: {catalog}")
print(f"data_source: {data_source}")

### Step 5: Configure Paths and Load Bronze Data
Sets up S3 paths and reads data from the Bronze layer.

In [ ]:
S3_BASE        = f"s3://healthcare-claims-lakehouse/raw/{data_source}"
CHECKPOINT_BASE = f"s3://healthcare-claims-lakehouse/checkpoints/{data_source}"
TARGET_SCHEMA  = f"{catalog}.silver"

# Define bronze_schema since utilities %run failed
bronze_schema = 'bronze'

In [ ]:
df_bronze = spark.sql(f'select * from {catalog}.{bronze_schema}.{data_source}')

### Step 6: Preview Bronze Data
Displays sample records and schema to understand the data structure.

In [ ]:
display(df_bronze.limit(30))

In [ ]:
df_bronze.printSchema()

### Step 7: Drop Metadata Columns
Removes Auto Loader metadata columns that are not needed in Silver layer.

In [ ]:
df_bronze = df_bronze.drop(
    "_rescued_data",
    "file_name",
    "file_size",
    "read_timestamp"
)

### Step 8: Standardize Invalid Values
Replaces common invalid value placeholders (empty string, NULL, NA, *) with proper NULL values.

In [ ]:
df_bronze = df_bronze.select([
    F.when(F.col(c).isin("", "NULL", "NA", "*"), None)
     .otherwise(F.col(c))
     .alias(c)
    for c in df_bronze.columns
])

### Step 9: Inspect Numeric Columns
Checks sample values in numeric columns to understand data quality.

In [ ]:
df_bronze.select("Tot_Clms", "Tot_Drug_Cst").show(5)

### Step 10: Initial Type Casting
Casts key numeric columns to appropriate types.

In [ ]:
df_bronze = df_bronze \
    .withColumn("Tot_Benes", F.col("Tot_Benes").cast("int")) \
    .withColumn("Tot_Drug_Cst", F.col("Tot_Drug_Cst").cast("double")) \
    .withColumn("Tot_30day_Fills", F.col("Tot_30day_Fills").cast("double"))

In [ ]:
df_bronze.select("Tot_Clms").show(10)

### Step 11: Safe Type Casting with try_cast
Uses `try_cast` for safe casting - returns NULL if conversion fails instead of throwing errors.

**INT columns**: Strict count values (provider/beneficiary counts)
**DOUBLE columns**: All numeric measurement fields (costs, claims, rates)

In [ ]:
# INT columns (STRICT counts ONLY — be careful)
int_cols = [
    "Tot_Benes",
    "Bene_Feml_Cnt",
    "Bene_Male_Cnt"
]

# DOUBLE columns (MOST numeric fields go here safely)
double_cols = [
    "Tot_Clms",   # moved here
    "Tot_30day_Fills",
    "Tot_Drug_Cst",
    "Tot_Day_Suply",
    "Opioid_Tot_Clms",
    "Opioid_Tot_Drug_Cst",
    "Opioid_Prscrbr_Rate",
    "Bene_Avg_Age",
    "Bene_Avg_Risk_Scre"
]

# SAFE CASTING
for c in int_cols:
    if c in df_bronze.columns:
        df_bronze = df_bronze.withColumn(c, F.expr(f"try_cast({c} as int)"))

for c in double_cols:
    if c in df_bronze.columns:
        df_bronze = df_bronze.withColumn(c, F.expr(f"try_cast({c} as double)"))

In [ ]:
df_bronze.printSchema()

### Step 12: Column Standardization
Converts all column names to lowercase for consistency.

In [ ]:
df_bronze  = df_bronze.toDF(*[c.lower() for c in df_bronze.columns])

In [ ]:
display(df_bronze.limit(5))

### Step 13: Text Column Standardization
Trims whitespace and converts to uppercase for key text columns to ensure consistency.

In [ ]:
df_bronze.select('prscrbr_city').show(6)

In [ ]:
display(df_bronze.withColumn(
    "prscrbr_city",
    F.upper(F.trim(F.col("prscrbr_city")))
).limit(6))

In [ ]:
text_cols = [
    "prscrbr_city",
    "prscrbr_state_abrvtn",
    "prscrbr_cntry",
    "brnd_name",
    "gnrc_name"
]

for c in text_cols:
    if c in df_bronze.columns:
        df_bronze = df_bronze.withColumn(
            c,
            F.upper(F.trim(F.col(c)))
        )

In [ ]:
df_bronze.select(
    "prscrbr_city",
    "prscrbr_state_abrvtn",
    "brnd_name"
).show(10, False)

### Step 14: Check and Remove Duplicates
Identifies duplicate records based on key columns and removes them.

In [ ]:
df_bronze.groupBy("prscrbr_npi", "brnd_name", "gnrc_name") \
    .count() \
    .filter("count > 1") \
    .show()

In [ ]:
df_bronze = df_bronze.dropDuplicates([
    "prscrbr_npi",
    "brnd_name",
    "gnrc_name"
])

### Step 15: Data Modeling - Silver Layer Design

## 🥈 Silver Layer Design — Data Modeling Strategy

### ⚔️ Think Like a Data Engineer

Instead of keeping one large, unstructured dataset, the data is split into **logical components** based on its purpose.

This improves:
- Query performance
- Data clarity
- Scalability
- Maintainability

---

## 🧠 Data Breakdown

The dataset is divided into **3 core components**:

---

### 1️⃣ Provider Information (Dimension Table)

**Description:**
Contains static or slowly changing attributes about providers.

**Columns:**
- `prscrbr_npi`
- `prscrbr_first_name`
- `prscrbr_last_org_name`
- `prscrbr_city`
- `prscrbr_state_abrvtn`
- `prscrbr_cntry`

**Why?**
- Represents **entities (providers)**
- Used for joins and filtering
- Avoids redundancy in fact tables

---

### 2️⃣ Drug Information (Dimension Table)

**Description:**
Contains descriptive information about drugs.

**Columns:**
- `brnd_name`
- `gnrc_name`

**Why?**
- Separates **categorical drug data**
- Enables clean grouping and analysis
- Reduces duplication

---

### 3️⃣ Metrics (Fact Table)

**Description:**
Contains measurable values used for analysis.

**Columns (examples):**
- `tot_clms`
- `tot_drug_cst`
- `tot_benes`
- `opioid_tot_clms`
- `opioid_tot_drug_cst`

**Why?**
- Central table for analytics
- Stores numerical and aggregated data
- Links to dimension tables via keys

---

## 🔗 Data Relationships

- `prscrbr_npi` → links to Provider Dimension
- `brnd_name`, `gnrc_name` → link to Drug Dimension

This forms a **star schema-like structure**:
- Fact table in center
- Dimension tables around it

---

## 🚀 Benefits of This Design

- ✅ Faster queries (smaller, focused tables)
- ✅ Cleaner schema (no unnecessary duplication)
- ✅ Scalable for large datasets
- ✅ Industry-standard data modeling approach

---

## 💡 Key Insight

> Instead of asking "How do I store data?",
> think: **"How will this data be used?"**

This shift is what separates:
- ❌ Basic data processing
- ✅ Real data engineering systems

### Step 16: Create Provider Dimension Table
Extracts provider-specific columns into a dimension table.

In [ ]:
provider_df = df_bronze.select(
    "prscrbr_npi",
    "prscrbr_first_name",
    "prscrbr_last_org_name",
    "prscrbr_city",
    "prscrbr_state_abrvtn",
    "prscrbr_cntry"
).dropDuplicates()

### Step 17: Create Fact Table
Extracts metric columns for the fact table.

In [ ]:
fact_df = df_bronze.select(
    "prscrbr_npi",
    "brnd_name",
    "prscrbr_state_abrvtn",
    "gnrc_name",
    "tot_clms",
    "tot_drug_cst",
    "tot_benes",
    "opioid_tot_clms",
    "opioid_tot_drug_cst"
)

In [ ]:
provider_df.show(5)

In [ ]:
fact_df.show(5)

### Step 18: Add Provider ID and Write Provider Table
Adds a unique provider_id and writes to Delta table with Change Data Feed enabled.

In [ ]:
provider_df = provider_df.dropDuplicates(["prscrbr_npi"]) \
    .withColumn("provider_id", F.monotonically_increasing_id())

In [ ]:
provider_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(f"{catalog}.{silver_schema}.provider")

In [ ]:
print("bronze:", df_bronze.count())
print("provider:", provider_df.count())

### Step 19: Write Fact Prescriptions Table
Writes the fact table partitioned by state for optimized querying.

In [ ]:
fact_df.show(10)

In [ ]:
fact_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .partitionBy("prscrbr_state_abrvtn") \
    .saveAsTable(f"{catalog}.{silver_schema}.fact_prescriptions")
